# 11.7 Sort Merge Joins

## Learning Objectives

By the end of this lesson you will be able to:

- Understand what a Sort Merge Join is
- Learn why Spark uses Sort Merge Join
- Understand the execution flow
- Learn how large datasets are joined
- Identify Sort Merge Join in execution plans
- Understand performance implications

> **Core Rule:** Sort Merge Join is the most common join strategy used for large-scale distributed data processing.

## Setup: Initialize Spark & Mock Data

Let's start our local Spark Session and generate two "Large" datasets to demonstrate how Spark handles joins when broadcasting isn't an option.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Sort_Merge_Joins_Deep_Dive") \
    .master("local[*]") \
    .getOrCreate()

# Create Large Table 1: Orders (e.g., 100,000 rows for local testing)
large_orders = spark.range(1, 100000).withColumnRenamed("id", "order_id") \
                    .withColumn("customer_id", (F.rand() * 50000).cast("int")) \
                    .withColumn("amount", F.rand() * 100)

# Create Large Table 2: Customers (e.g., 50,000 rows for local testing)
large_customers = spark.range(1, 50000).withColumnRenamed("id", "customer_id") \
                       .withColumn("signup_date", F.current_date())

print("Spark Session Initialized and Mock Large Data Created!")

# Why Learn Sort Merge Joins?

In the previous lesson, we learned that Broadcast Join works well when one table is small.

But what happens when:
- Both tables are large?
- Neither table fits in memory?
- Broadcasting is impossible?

Spark needs a different strategy. This is where **Sort Merge Join** becomes important.

# Real-World Scenario

Imagine an e-commerce company.

**Table 1:** Orders (500 Million Rows)
**Table 2:** Customers (100 Million Rows)

Clearly: Neither table can be broadcast safely without causing an OutOfMemory error.

Spark must use a strategy designed for large datasets. That strategy is often Sort Merge Join.

# What is a Sort Merge Join?

Sort Merge Join is a join strategy that:

1. Redistributes data by join key
2. Sorts data within partitions
3. Merges matching records

Unlike Broadcast Join, both datasets participate in the join process.

<h3>Sort Merge Join Overview</h3>

<img src="../assets/Module_11/11_07_01.png" alt="Sort Merge Join Overview" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark Sort Merge Join overview showing two large datasets entering shuffle phase, sorting phase and merge phase. Educational architecture diagram with arrows and labeled stages.</i></p>

# Why Spark Uses Sort Merge Join

Sort Merge Join is designed for:
- Large datasets
- Distributed environments
- High scalability

Although it is more expensive than Broadcast Join, it works reliably when both datasets are massive.

# High-Level Execution Flow

The execution flow consists of three major phases:

- **Phase 1:** Shuffle
- **Phase 2:** Sort
- **Phase 3:** Merge

Understanding these phases helps explain why Sort Merge Join can be expensive.

<h3>Three Phases of Sort Merge Join</h3>

<img src="../assets/Module_11/11_07_02.png" alt="Three Phases" width="75%">

<p><i><b>Image Prompt:</b> Three-stage Spark Sort Merge Join workflow. Shuffle phase, Sort phase and Merge phase shown sequentially with large datasets moving through the pipeline. Beginner-friendly infographic.</i></p>

# Phase 1: Shuffle

The first step is data redistribution.

Spark groups records with the same join key into the same partition.

**Example:** `Customer ID = 101`

All matching records must eventually reach the same partition. This requires data movement across the cluster.

# Why Shuffle Is Needed

Suppose Customer 101 exists on:
- Executor A (Orders data)
- Executor B (Orders data)
- Executor C (Customers data)

Spark cannot perform a join until related records are brought together.

Shuffle ensures matching records reach the same partition so they can be joined locally.

<h3>Shuffle Phase</h3>

<img src="../assets/Module_11/11_07_03.png" alt="Shuffle Phase" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark shuffle process for Sort Merge Join. Records with identical join keys moving from multiple executors into common partitions. Educational cluster visualization.</i></p>

# Phase 2: Sort

After shuffle, records within each partition are sorted by the join key.

**Before Sort:**
`101, 104, 102, 103`

**After Sort:**
`101, 102, 103, 104`

Sorting prepares the data for efficient matching.

# Why Sorting Helps

Once records are sorted, Spark can scan through datasets extremely efficiently.

Instead of repeatedly searching (looping) for matching records, it can move through both datasets in order using a simple two-pointer algorithm.

This drastically improves matching performance.

<h3>Sorting Within Partitions</h3>

<img src="../assets/Module_11/11_07_04.png" alt="Sorting Phase" width="75%">

<p><i><b>Image Prompt:</b> Spark partitions before sorting and after sorting by join key. Educational comparison showing ordered records enabling efficient joins.</i></p>

# Phase 3: Merge

The final phase is merging.

Spark compares sorted records from both datasets. Matching keys are combined.

**Example:** `Orders.CustomerID = Customers.CustomerID`

Matching rows are joined together into a single wide row.

# Merge Process Example

**Dataset A:** `101, 102, 103`

**Dataset B:** `101, 102, 105`

**Matches:** `101, 102`

Non-matching records are skipped (in an Inner Join).

This process is highly efficient once data is sorted.

<h3>Merge Phase</h3>

<img src="../assets/Module_11/11_07_05.png" alt="Merge Phase" width="75%">

<p><i><b>Image Prompt:</b> Sort Merge Join merge phase showing two sorted datasets being scanned together and matching records joined. Clean educational architecture diagram.</i></p>

# Putting It All Together

Sort Merge Join execution pipeline:

```text
Large Dataset A          Large Dataset B
        ↓                       ↓
     Shuffle                 Shuffle
        ↓                       ↓
       Sort                    Sort
        ↓                       ↓
        +------- Merge ---------+
                   ↓
                 Result
```

In [ ]:
# Let's perform a join. 
# We turn OFF Auto Broadcast Join so Spark is FORCED to use a Sort Merge Join.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("Executing Join of Large Datasets...")
# Perform the join
joined_df = large_orders.join(large_customers, "customer_id")

# Trigger an action to run the SMJ
print(f"Total matched records: {joined_df.count():,}")

# Re-enable broadcast for best practices moving forward
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

<h3>Complete Sort Merge Join Flow</h3>

<img src="../assets/Module_11/11_07_06.png" alt="Complete Flow" width="75%">

<p><i><b>Image Prompt:</b> End-to-end Sort Merge Join execution flow diagram. Large Dataset A and Large Dataset B moving through Shuffle, Sort and Merge stages before producing final output.</i></p>

# Why Is Sort Merge Join Expensive?

Sort Merge Join introduces multiple costs:

- **Network Transfer:** Moving data across nodes.
- **Shuffle Operations:** Writing intermediate files to disk.
- **Sorting Overhead:** In-memory sorting algorithms.
- **Disk I/O:** Reading and writing shuffle files.
- **CPU Consumption:** Serialization, sorting, and merging.

These costs increase execution time.

# Cost 1: Shuffle

Both datasets participate in redistribution.

Large data movement across machines can create:
- Network bottlenecks
- Long stage durations
- Resource contention

# Cost 2: Sorting

Sorting millions of records requires:
- CPU resources
- Memory resources
- Temporary storage (if memory fills up, sorting spills to disk)

Sorting can become a significant performance bottleneck.

<h3>Cost of Sort Merge Join</h3>

<img src="../assets/Module_11/11_07_07.png" alt="Cost Breakdown" width="75%">

<p><i><b>Image Prompt:</b> Cost breakdown infographic showing Shuffle Cost, Sorting Cost, Network Cost and CPU Cost contributing to Sort Merge Join execution time.</i></p>

# Advantages of Sort Merge Join

Benefits include:
- Handles very large datasets
- Scales well across clusters
- Reliable for distributed processing
- Works even when broadcasting is impossible

This is why Spark frequently selects it as the default for large tables.

# Limitations of Sort Merge Join

Challenges include:
- Heavy shuffle
- Sorting overhead
- Increased execution time
- Resource-intensive processing

It is usually slower than Broadcast Join, but much safer for big data.

# Large Dataset Join Example

**Orders Table:** 500 Million Rows

**Customers Table:** 100 Million Rows

**Broadcast Join:** Not practical (will crash cluster memory).

**Sort Merge Join:** Designed specifically for this scenario.

<h3>Large Dataset Join Scenario</h3>

<img src="../assets/Module_11/11_07_08.png" alt="Large Dataset Scenario" width="75%">

<p><i><b>Image Prompt:</b> Spark cluster joining two massive datasets using Sort Merge Join. Hundreds of millions of records processed through shuffle and sorting phases. Educational big data visualization.</i></p>

# Sort Merge Join in Execution Plans

Execution Plans reveal the selected join strategy. 

Let's look at the physical plan of the SMJ we ran earlier to see the `SortMergeJoin`, `Sort`, and `Exchange` operators.

In [ ]:
print("Physical Plan for our Sort Merge Join:")
print("-"*50)
joined_df.explain()
print("-"*50)
print("Read the plan from Bottom to Top:")
print("1. Exchange (Shuffle Phase)")
print("2. Sort (Sort Phase)")
print("3. SortMergeJoin (Merge Phase)")

# Example Physical Plan

You saw above:

SortMergeJoin
    ↓
Sort
    ↓
Exchange

This tells us:
- Data redistribution is required (`Exchange`)
- Data is ordered within partitions (`Sort`)
- Sort Merge Join is being used (`SortMergeJoin`)

<h3>SortMergeJoin Operator</h3>

<img src="../assets/Module_11/11_07_09.png" alt="SortMergeJoin Operator" width="75%">

<p><i><b>Image Prompt:</b> Spark Physical Plan highlighting SortMergeJoin and Exchange operators. Educational execution plan visualization demonstrating shuffle and sorting stages.</i></p>

# Comparing with Broadcast Join

| Feature | Broadcast Join | Sort Merge Join |
|----------|----------|----------|
| **Shuffle** | Minimal | High |
| **Sorting** | No | Yes |
| **Best For** | Small Tables | Large Tables |
| **Speed** | Faster | Slower |
| **Scalability** | Limited by Memory | High |

Both strategies have extremely important use cases.

# Key Takeaways

- Sort Merge Join is Spark's common strategy for large dataset joins.
- Execution occurs in three phases:
  - **Shuffle**
  - **Sort**
  - **Merge**
- It scales well for distributed processing.
- It is more expensive than Broadcast Join because of Network and CPU overhead.
- Physical Plans reveal `Exchange`, `Sort`, and `SortMergeJoin` operators.
- Understanding Sort Merge Join is essential for Spark performance tuning.

---

# Next Lesson: 11.8 Shuffle Hash Joins

In the next lesson we will learn:
- Internal Working of Shuffle Hash Joins
- Trade-offs compared to Sort Merge

This will complete our study of Spark's major join strategies.